# Análise de Radiossondagens — Seminário de Sondagem (FSC7116 Mesoescala · UFSC)

**Prof. Reinaldo Haas** — versão revisada do notebook *Seminario_plot_sounding.ipynb*.

Este notebook faz, para **uma ou mais sondagens**:

1. Download dos dados (Universidade de Wyoming) **ou** leitura de arquivo local — **nunca** substitui por dados de exemplo.
2. **Controle de qualidade (QC)** dos níveis da sondagem.
3. Índices termodinâmicos e cinemáticos com **convenções do Hemisfério Sul**.
4. Skew-T + hodógrafo, perfis de θ, θe, θes, N² e estabilidade estática.
5. Matrizes de diferença de temperatura de densidade de **Kerry Emanuel** (`wyoming.f`, MIT).
6. Satélite (opcional): imagem IR real GridSat-B1 (0,07°).
7. Carta sinótica (opcional) com **reanálise real** (NCEP/NCAR R1).
8. Exporta **todos** os números para `saida/metricas.json`.

> **Regra de ouro:** nenhum número pode ser digitado à mão em figura, slide ou texto.
> Tudo o que for dito na apresentação deve vir de `saida/metricas.json` ou de uma figura gerada aqui.

## O que foi corrigido em relação ao notebook original

| # | Erro no notebook original | Correção |
|---|---|---|
| 1 | Se o download falhava, carregava **silenciosamente a sondagem de exemplo de Oklahoma (OUN)** e seguia a análise como se fosse a estação pedida | Sem dados de exemplo: tenta Wyoming (siphon → URL direta) → arquivo local salvo; se nada existir, **para com erro** |
| 2 | `mpio.read_soundings(...)` — função que não existe no MetPy | Removida |
| 3 | `lifted_index(p, T, Td)` — o 3º argumento deve ser o **perfil da parcela**, não Td | `lifted_index(p, T, perfil_sb)` |
| 4 | `shade_cape(p, T, Td)` / `shade_cin(p, T, Td)` — sombreava usando Td como se fosse a parcela | Usa o perfil da parcela |
| 5 | SRH calculada **sem movimento da tempestade** (helicidade relativa ao solo), pegando só a parte positiva `[0]` | SRH total `[2]` relativa ao movimento de Bunkers; no HS usa-se o **left-mover** e SRH ciclônica é **negativa** |
| 6 | "Effective SRH" = SRH 0–1 km total; "Effective shear" = LFC→EL da parcela MU | Removidos (definições erradas). Ficam SRH e cisalhamento em camadas fixas, bem definidos |
| 7 | Alturas MSL usadas onde o MetPy espera **altura acima do solo (AGL)** | `h = z − z_sfc` |
| 8 | Parcela "ML" = média simples de T e Td em 50 hPa, levantada a partir de p[0] | `mixed_layer_cape_cin` (θ e r médios, camada de 100 hPa) |
| 9 | "MESH proxy" com fórmula inventada ("may not be meteorologically accurate") | Removido |
| 10 | SigSevere, BRN, SCP, STP com definições incorretas | Removidos, exceto STP de camada fixa (Thompson et al. 2012), com a adaptação ao HS explícita |
| 11 | Perfis de θ/N usavam altura da **atmosfera padrão** em vez da altura observada | Usa a altura observada (HGHT) |
| 12 | Estabilidade estática rotulada "K/km" (unidade errada) | −(T/θ)∂θ/∂p em **K/hPa** |
| 13 | Freq. de Brunt-Väisälä N plotada só onde N² > 0 (NaN na camada instável) | Plota **N²**, com sinal |
| 14 | URL do Fortran fixa para **outra estação e outra data** (83952, 2026-10-05) diferentes do cabeçalho | `sounding.txt` escrito pelo notebook **a partir da mesma sondagem** analisada |
| 15 | `pd.read_fwf('tdifrev.out')` usa a 1ª linha de dados como cabeçalho (perde um nível; daí o `pp.iloc[0:-1]`) | `np.loadtxt` |
| 16 | `plt.show()` antes de `plt.savefig()` → PNG em branco | `savefig` antes de `show` |
| 17 | Mapa de cores `jet` sem zero centralizado nas matrizes de Emanuel | Divergente centrado em 0 K; região não física (nível final abaixo da origem) mascarada |
| 18 | Interpretação em texto fixo ("16 de janeiro de 2025", "chegando a 3 K") que não corresponde aos dados | Removida: a interpretação é do aluno, citando `metricas.json` |
| 19 | Sem controle de qualidade | QC: Td > T, camadas superadiabáticas acima da superfície, saltos de θe, consistência hidrostática |

## 1. Instalação e importações

In [ ]:
import sys, os, subprocess, json, re, shutil, warnings
from datetime import datetime

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'metpy', 'siphon', 'cartopy', 'netCDF4'], check=True)
    # Repositório com as sondagens salvas, wyoming.f e arquivos de satélite
    if not os.path.exists('tarefa-meso'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/reinaldohaas/tarefa-meso.git'], check=True)
    os.chdir('tarefa-meso')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import metpy
import metpy.calc as mpcalc
from metpy.plots import SkewT, Hodograph
from metpy.units import units

print('MetPy', metpy.__version__, '| diretório:', os.getcwd())

## 2. Configuração

Edite **apenas esta célula**. Cada caso é uma sondagem: código WMO da estação e data/hora (UTC).
As sondagens baixadas são salvas em `DIR_DADOS` como `sounding_AAAAMMDD_HHZ.txt` (texto original de Wyoming).

In [ ]:
CASOS = [
    dict(estacao='83971', nome='SBPA Porto Alegre', datahora='1995-12-12 12:00'),
    dict(estacao='83971', nome='SBPA Porto Alegre', datahora='1995-12-23 12:00'),
    dict(estacao='83971', nome='SBPA Porto Alegre', datahora='1995-12-24 12:00'),
]

DIR_DADOS = 'metpack'        # onde ficam/serão salvas as sondagens em texto
DIR_SAIDA = 'saida'          # figuras e metricas.json
TENTAR_DOWNLOAD = True       # False = usar só os arquivos locais já salvos
FONTE_WYOMING = 'FM35'       # 'FM35' (mensagem TEMP; dados antigos como 1995) ou 'BUFR' (dados recentes)
CAMADA_ML_HPA = 100          # espessura da camada de mistura (hPa) para MLCAPE
P_TOPO = 300                 # topo (hPa) dos GRÁFICOS de perfil (Skew-T, hodógrafo, θ/N²/S); os cálculos usam a sondagem inteira
P_TOPO_EMANUEL = 100         # topo (hPa) das matrizes de Emanuel — mostra até onde a flutuabilidade zera
# O wyoming.f original limita as matrizes médias a no mínimo -4 K (TPDBAR = MAX(TPDBAR,-4.0)).
# None = sem esse corte (mostra toda a flutuabilidade negativa, como o wyoming.py); -4.0 = idêntico ao original.
CORTE_EMANUEL_K = None

# Teste de sensibilidade (opcional): níveis a excluir. Os valores oficiais usam a sondagem completa.
# Exemplo: {'19951224_12': [942.5, 925.0, 910.5]}
EXCLUIR_NIVEIS = {}

# Opcionais (precisam de internet)
FAZER_SATELITE = True        # GridSat-B1: temperatura de brilho IR, 0,07°
FAZER_REANALISE = True       # carta sinótica NCEP/NCAR R1 via OPeNDAP (PSL/NOAA)

os.makedirs(DIR_SAIDA, exist_ok=True)
for c in CASOS:
    c['dt'] = datetime.strptime(c['datahora'], '%Y-%m-%d %H:%M')
    c['chave'] = c['dt'].strftime('%Y%m%d_%H')
    c['rotulo'] = f"{c['nome']} — {c['dt']:%d/%m/%Y %HZ}"
[c['chave'] for c in CASOS]

## 3. Obtenção dos dados

Ordem de tentativa: **URL direta** de Wyoming (formato TEXT:LIST, que inclui os índices calculados pelo Wyoming) → **siphon**
(WyomingUpperAir, só a tabela) → **arquivo local** em `DIR_DADOS`.
Se nenhuma funcionar, a célula **para com erro**. A fonte usada fica registrada em `metricas.json`.

In [ ]:
COLUNAS_WYO = ['p', 'z', 'T', 'Td', 'rh', 'mixr', 'drct', 'sknt', 'thta', 'thte', 'thtv']

def le_texto_wyoming(texto):
    # Lê a tabela TEXT:LIST de Wyoming (colunas de largura fixa = 7 caracteres).
    linhas = re.sub(r'<[^>]+>', '', texto).splitlines()
    i0 = next(i for i, l in enumerate(linhas) if 'PRES' in l and 'HGHT' in l)
    # dados começam após a 2ª linha tracejada depois do cabeçalho
    tracos = [i for i in range(i0, min(i0 + 6, len(linhas))) if linhas[i].strip().startswith('---')]
    inicio = tracos[-1] + 1
    regs = []
    for l in linhas[inicio:]:
        if not l.strip() or not re.match(r'^\s*\d', l):
            break
        campos = [l[k*7:(k+1)*7].strip() for k in range(11)]
        regs.append([float(x) if x else np.nan for x in campos])
    df = pd.DataFrame(regs, columns=COLUNAS_WYO)
    return df[['p', 'z', 'T', 'Td', 'drct', 'sknt']]

def arquivo_local(caso):
    return os.path.join(DIR_DADOS, f"sounding_{caso['dt']:%Y%m%d_%H}Z.txt")

def url_wyoming(caso, tipo):
    return ('https://weather.uwyo.edu/wsgi/sounding?datetime='
            f"{caso['dt']:%Y-%m-%d}%20{caso['dt']:%H:%M:%S}&id={caso['estacao']}&src={FONTE_WYOMING}&type={tipo}")

def arquivo_indices(caso):
    return os.path.join(DIR_DADOS, f"indices_{caso['dt']:%Y%m%d_%H}Z.txt")

def _num_ou_none(x):
    m = re.fullmatch(r'\s*(-?\d+(?:\.\d+)?)\s*(?:\S+)?\s*', str(x))
    return float(m.group(1)) if m else None

def le_indices_wyoming(texto):
    # Índices calculados pelo próprio Wyoming (página type=INDICES ou bloco "Station information and sounding indices").
    # Aceita linhas "nome: valor" ou tabela HTML de 2 colunas. Devolve {nome: valor}.
    if not texto:
        return {}
    ind = {}
    limpo = re.sub(r'<[^>]+>', '\n', texto)
    for l in limpo.splitlines():
        m = re.match(r'^\s*([A-Za-z][^:]{1,80}?):\s*(-?\d+(?:\.\d+)?)\s*$', l)
        if m and not m.group(1).strip().startswith(('Station', 'Observation')):
            ind[m.group(1).strip()] = float(m.group(2))
    if not ind and '<t' in texto.lower():
        try:
            from io import StringIO
            for tab in pd.read_html(StringIO(texto)):
                if tab.shape[1] >= 2:
                    for nome, val in tab.iloc[:, :2].itertuples(index=False):
                        v = _num_ou_none(val)
                        if isinstance(nome, str) and v is not None:
                            ind[nome.strip().rstrip(':')] = v
        except Exception:
            pass
    return ind

def obtem_indices_wyoming(caso, texto_sondagem=''):
    # 1) página type=INDICES (salva em DIR_DADOS); 2) arquivo salvo; 3) bloco dentro do texto da sondagem
    if TENTAR_DOWNLOAD:
        try:
            import requests
            r = requests.get(url_wyoming(caso, 'INDICES'), timeout=60); r.raise_for_status()
            ind = le_indices_wyoming(r.text)
            if ind:
                with open(arquivo_indices(caso), 'w', encoding='utf-8') as f:
                    f.write(r.text)
                return ind, url_wyoming(caso, 'INDICES')
        except Exception as e:
            print('   (índices do Wyoming não baixados:', type(e).__name__, e, ')')
    if os.path.exists(arquivo_indices(caso)):
        ind = le_indices_wyoming(open(arquivo_indices(caso), encoding='utf-8', errors='ignore').read())
        if ind:
            return ind, arquivo_indices(caso)
    ind = le_indices_wyoming(texto_sondagem)
    return ind, ('bloco no texto da sondagem' if ind else None)

def obtem_sondagem(caso):
    # Devolve (tabela, fonte, texto original). O texto original guarda os índices calculados pelo Wyoming.
    erros = []
    if TENTAR_DOWNLOAD:
        try:
            import requests
            url = url_wyoming(caso, 'TEXT:LIST')
            r = requests.get(url, timeout=60); r.raise_for_status()
            df = le_texto_wyoming(r.text)
            if len(df) < 5:
                raise ValueError('resposta sem tabela de sondagem')
            with open(arquivo_local(caso), 'w', encoding='utf-8') as f:
                f.write(r.text)
            return df, f'Univ. Wyoming ({url})', r.text
        except Exception as e:
            erros.append(f'URL direta: {e}')
        try:
            from siphon.simplewebservice.wyoming import WyomingUpperAir
            d = WyomingUpperAir.request_data(caso['dt'], caso['estacao'])
            df = d.rename(columns={'pressure': 'p', 'height': 'z', 'temperature': 'T',
                                   'dewpoint': 'Td', 'direction': 'drct', 'speed': 'sknt'})
            return df[['p', 'z', 'T', 'Td', 'drct', 'sknt']], 'Univ. Wyoming (siphon; sem índices do Wyoming)', ''
        except Exception as e:
            erros.append(f'siphon: {e}')
    if os.path.exists(arquivo_local(caso)):
        with open(arquivo_local(caso), encoding='utf-8', errors='ignore') as f:
            texto = f.read()
        return le_texto_wyoming(texto), f'arquivo local {arquivo_local(caso)} (texto original de Wyoming)', texto
    raise RuntimeError(f"Sem dados para {caso['rotulo']}. Tentativas: {erros}. "
                       'NÃO há dados de exemplo substitutos — obtenha a sondagem real.')

def limpa(df):
    n0 = len(df)
    df = (df.dropna(subset=['p', 'z', 'T', 'Td', 'drct', 'sknt'])
            .sort_values('p', ascending=False)
            .drop_duplicates('p')
            .reset_index(drop=True))
    return df, n0 - len(df)

for c in CASOS:
    bruto, c['fonte'], texto = obtem_sondagem(c)
    c['indices_wyoming'], c['fonte_indices_wyoming'] = obtem_indices_wyoming(c, texto)
    c['df_original'], c['n_descartados'] = limpa(bruto)
    print(f"{c['rotulo']}: {len(c['df_original'])} níveis completos "
          f"({c['n_descartados']} níveis com dado faltante descartados) — fonte: {c['fonte']}")
    if c['indices_wyoming']:
        print(f"   índices do Wyoming: {len(c['indices_wyoming'])} lidos de {c['fonte_indices_wyoming']}")
    else:
        print('   ⚠ índices do Wyoming indisponíveis — rode com TENTAR_DOWNLOAD = True ou salve a página')
        print('     ', url_wyoming(c, 'INDICES'), '→', arquivo_indices(c))

## 4. Controle de qualidade (QC)

Testes aplicados a cada par de níveis vizinhos:

* **Td > T** (fisicamente impossível);
* **camada superadiabática acima de 50 hPa da superfície** (Γ > 10 K/km, camadas com pelo menos 50 m) — comum junto à superfície de dia; em altitude é raro e merece ser examinado;
* **pico isolado de θe**: nível que difere mais de 8 K dos **dois** vizinhos;
* **consistência hidrostática** (até 100 hPa): espessura observada × espessura hipsométrica com Tv (diferença > 5 %).

O QC **só aponta** níveis que merecem atenção — não significa que estejam errados. **Os valores oficiais são os
calculados com a sondagem completa, como publicada por Wyoming.** `EXCLUIR_NIVEIS` (célula 2) existe apenas para
**testes de sensibilidade**: se usado, os índices são calculados com e sem os níveis indicados e os dois aparecem em
`metricas.json`.

In [ ]:
def controle_qualidade(df):
    p = df['p'].values; z = df['z'].values; T = df['T'].values; Td = df['Td'].values
    the = mpcalc.equivalent_potential_temperature(p*units.hPa, T*units.degC, Td*units.degC).m
    tv = mpcalc.virtual_temperature_from_dewpoint(p*units.hPa, T*units.degC, Td*units.degC).to('K').m
    flags = []
    for i in range(len(df)):
        if Td[i] > T[i] + 0.05:
            flags.append((p[i], 'Td > T', f'T={T[i]:.1f} Td={Td[i]:.1f} °C'))
    for i in range(len(df) - 1):
        dz = z[i+1] - z[i]
        if dz <= 0:
            flags.append((p[i+1], 'altura não crescente', f'Δz={dz:.0f} m')); continue
        gama = -(T[i+1] - T[i]) / dz * 1000
        if gama > 10.0 and p[i] < p[0] - 50 and dz >= 50:
            flags.append((p[i+1], 'superadiabática em altitude',
                          f'{p[i]:.1f}→{p[i+1]:.1f} hPa: Γ={gama:.1f} K/km'))
        dz_hip = 287.05 * 0.5*(tv[i]+tv[i+1]) / 9.80665 * np.log(p[i]/p[i+1])
        if abs(dz - dz_hip) > 0.05*dz_hip and dz_hip > 50 and p[i+1] >= 100:
            flags.append((p[i+1], 'inconsistência hidrostática',
                          f'Δz obs={dz:.0f} m, hipsométrica={dz_hip:.0f} m'))
    for i in range(1, len(df) - 1):
        if (the[i] - the[i-1] > 8 and the[i] - the[i+1] > 8) or (the[i-1] - the[i] > 8 and the[i+1] - the[i] > 8):
            flags.append((p[i], 'pico de θe', f'θe={the[i]:.1f} K (vizinhos {the[i-1]:.1f} / {the[i+1]:.1f})'))
    return pd.DataFrame(flags, columns=['p_hPa', 'teste', 'detalhe'])

for c in CASOS:
    c['qc'] = controle_qualidade(c['df_original'])
    print(f"\n=== QC {c['rotulo']} ===")
    print('nenhum nível suspeito' if c['qc'].empty else c['qc'].to_string(index=False))
    excl = EXCLUIR_NIVEIS.get(c['chave'], [])
    c['niveis_excluidos'] = list(excl)
    c['df'] = c['df_original'][~c['df_original']['p'].isin(excl)].reset_index(drop=True)
    if excl:
        print(f'>> níveis EXCLUÍDOS por decisão em EXCLUIR_NIVEIS: {excl}')

## 5. Índices

Convenções:

* Alturas **acima do solo** (AGL).
* Movimento de tempestade de **Bunkers** (Bunkers et al. 2000). No **Hemisfério Sul** a supercélula dominante é a **left-mover (LM)**,
  e a helicidade **ciclônica é negativa**. Reportamos SRH com LM (principal) e com RM (comparação).
* **STP de camada fixa** (Thompson et al. 2003; forma SPC 2012) adaptado ao HS usando **−SRH₀₋₁(LM)**:
  STP = (SBCAPE/1500) · ((2000 − LCL)/1000) · (−SRH₀₋₁/150) · (BS₀₋₆/20), termo do LCL = 1 se LCL < 1000 m e 0 se > 2000 m,
  termo do cisalhamento = 0 se BS₀₋₆ < 12,5 m/s e limitado a 1,5 (30 m/s).
* **EHI₀₋₃** = SBCAPE · (−SRH₀₋₃ LM) / 160000.

Se algum cálculo falhar, o valor fica `NaN` **e o erro é mostrado** — nunca vira 0 em silêncio.

In [ ]:
def _num(q, unid=None):
    if q is None: return np.nan
    try:
        if unid is not None: q = q.to(unid)
        v = q.m if hasattr(q, 'm') else q
        v = np.asarray(v).item() if np.ndim(v) == 0 else np.asarray(v)[0]
        return float(v)
    except Exception:
        return np.nan

def interp_logp(p_alvo, p, y):
    # interpola y(p) linearmente em ln p
    if p_alvo is None or np.isnan(p_alvo): return np.nan
    return float(np.interp(np.log(p_alvo), np.log(p[::-1]), y[::-1]))

def calcula_indices(df):
    I, erros = {}, {}
    def tenta(nome, f):
        try:
            return f()
        except Exception as e:
            erros[nome] = f'{type(e).__name__}: {e}'
            return None

    p = df['p'].values * units.hPa
    T = df['T'].values * units.degC
    Td = df['Td'].values * units.degC
    z = df['z'].values * units.m
    h = z - z[0]
    u, v = mpcalc.wind_components(df['sknt'].values * units.knot, df['drct'].values * units.deg)
    pm, hm = df['p'].values, h.m

    perfil_sb = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    r = tenta('SB', lambda: mpcalc.surface_based_cape_cin(p, T, Td))
    I['SBCAPE_Jkg'], I['SBCIN_Jkg'] = (_num(r[0], 'J/kg'), _num(r[1], 'J/kg')) if r else (np.nan, np.nan)
    r = tenta('ML', lambda: mpcalc.mixed_layer_cape_cin(p, T, Td, depth=CAMADA_ML_HPA*units.hPa))
    I['MLCAPE_Jkg'], I['MLCIN_Jkg'] = (_num(r[0], 'J/kg'), _num(r[1], 'J/kg')) if r else (np.nan, np.nan)
    r = tenta('MU', lambda: mpcalc.most_unstable_cape_cin(p, T, Td))
    I['MUCAPE_Jkg'], I['MUCIN_Jkg'] = (_num(r[0], 'J/kg'), _num(r[1], 'J/kg')) if r else (np.nan, np.nan)
    mu = tenta('parcela MU', lambda: mpcalc.most_unstable_parcel(p, T, Td))
    I['MU_origem_hPa'] = _num(mu[0], 'hPa') if mu else np.nan

    lcl = tenta('LCL', lambda: mpcalc.lcl(p[0], T[0], Td[0]))
    I['LCL_hPa'] = _num(lcl[0], 'hPa') if lcl else np.nan
    lfc = tenta('LFC', lambda: mpcalc.lfc(p, T, Td, perfil_sb))
    I['LFC_hPa'] = _num(lfc[0], 'hPa') if lfc else np.nan
    el = tenta('EL', lambda: mpcalc.el(p, T, Td, perfil_sb))
    I['EL_hPa'] = _num(el[0], 'hPa') if el else np.nan
    for k in ['LCL', 'LFC', 'EL']:
        I[f'{k}_m_AGL'] = interp_logp(I[f'{k}_hPa'], pm, hm)

    I['LI_K'] = _num(tenta('LI', lambda: mpcalc.lifted_index(p, T, perfil_sb)), 'delta_degC')
    I['Showalter_K'] = _num(tenta('Showalter', lambda: mpcalc.showalter_index(p, T, Td)), 'delta_degC')
    I['K_index_C'] = _num(tenta('K', lambda: mpcalc.k_index(p, T, Td)), 'degC')
    I['TotalTotals_C'] = _num(tenta('TT', lambda: mpcalc.total_totals_index(p, T, Td)), 'delta_degC')
    I['PW_mm'] = _num(tenta('PW', lambda: mpcalc.precipitable_water(p, Td)), 'mm')
    I['SWEAT'] = _num(tenta('SWEAT', lambda: mpcalc.sweat_index(p, T, Td, df['sknt'].values*units.knot,
                                                               df['drct'].values*units.deg)))
    dc = tenta('DCAPE', lambda: mpcalc.downdraft_cape(p, T, Td))
    I['DCAPE_Jkg'] = _num(dc[0], 'J/kg') if dc else np.nan

    for d in (1, 3, 6):
        bs = tenta(f'shear0-{d}', lambda d=d: mpcalc.bulk_shear(p, u, v, height=h, depth=d*units.km))
        I[f'BS0_{d}km_ms'] = float(np.hypot(_num(bs[0], 'm/s'), _num(bs[1], 'm/s'))) if bs else np.nan
    bk = tenta('Bunkers', lambda: mpcalc.bunkers_storm_motion(p, u, v, h))
    if bk:
        rm, lm, med = [x.to('m/s') for x in bk]
        for nome, vec in (('RM', rm), ('LM', lm), ('media', med)):
            I[f'Bunkers_{nome}_u_ms'], I[f'Bunkers_{nome}_v_ms'] = float(vec[0].m), float(vec[1].m)
        for d in (1, 3):
            for nome, vec in (('LM', lm), ('RM', rm)):
                s = tenta(f'SRH0-{d} {nome}', lambda d=d, vec=vec: mpcalc.storm_relative_helicity(
                    h, u, v, depth=d*units.km, storm_u=vec[0], storm_v=vec[1]))
                I[f'SRH0_{d}km_{nome}_m2s2'] = _num(s[2], 'm^2/s^2') if s else np.nan

    # vento em níveis de interesse (para falar de JBN com número calculado)
    for niv in (925, 850, 700, 500, 250):
        if pm.min() <= niv <= pm.max():
            k = int(np.argmin(abs(pm - niv)))
            if abs(pm[k] - niv) < 1:
                I[f'vento_{niv}_dir'] = float(df['drct'].iloc[k]); I[f'vento_{niv}_kt'] = float(df['sknt'].iloc[k])
    baixo = df[df['p'] >= 700]
    k = baixo['sknt'].idxmax()
    I['vento_max_abaixo700_kt'] = float(baixo.loc[k, 'sknt'])
    I['vento_max_abaixo700_p_hPa'] = float(baixo.loc[k, 'p'])
    I['vento_max_abaixo700_dir'] = float(baixo.loc[k, 'drct'])

    # compostos (HS: SRH ciclônica negativa -> usa -SRH)
    srh1 = -I.get('SRH0_1km_LM_m2s2', np.nan); srh3 = -I.get('SRH0_3km_LM_m2s2', np.nan)
    I['EHI0_3_HS'] = I['SBCAPE_Jkg'] * srh3 / 160000
    lclh, bs6 = I['LCL_m_AGL'], I['BS0_6km_ms']
    t_lcl = 1.0 if lclh < 1000 else (0.0 if lclh > 2000 else (2000 - lclh) / 1000)
    t_bs = 0.0 if bs6 < 12.5 else min(bs6, 30.0) / 20
    I['STP_fixo_HS'] = (I['SBCAPE_Jkg'] / 1500) * t_lcl * (srh1 / 150) * t_bs

    perfis = dict(p=p, T=T, Td=Td, z=z, h=h, u=u, v=v, perfil_sb=perfil_sb)
    if mu:
        i_mu = int(np.argmin(abs(pm - mu[0].m)))
        perfis['p_mu'] = p[i_mu:]
        perfis['perfil_mu'] = mpcalc.parcel_profile(p[i_mu:], mu[1], mu[2]).to('degC')
    return I, erros, perfis

for c in CASOS:
    c['indices'], c['erros'], c['perfis'] = calcula_indices(c['df'])
    if c['niveis_excluidos']:
        c['indices_sem_exclusao'], _, _ = calcula_indices(c['df_original'])
    if c['erros']:
        print(f"⚠ ERROS DE CÁLCULO em {c['rotulo']}:")
        for k, e in c['erros'].items(): print('   ', k, '->', e)

tabela = pd.DataFrame({c['rotulo']: c['indices'] for c in CASOS})
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')
tabela

### 5.1 Conferência com os índices calculados pelo próprio Wyoming

Wyoming publica, para cada sondagem, uma página de **índices termodinâmicos** (Showalter, LI, SWEAT, K, Totals, CAPE,
CIN, LCL, LFC, EL, água precipitável…), por exemplo
`https://weather.uwyo.edu/wsgi/sounding?datetime=1995-12-24%2012:00:00&id=83971&src=FM35&type=INDICES`.
O notebook baixa essa página, salva em `metpack/indices_AAAAMMDD_HHZ.txt` e lê todos os índices. Eles servem de **guia para conferir** os cálculos deste notebook:

* diferenças pequenas são esperadas (Wyoming usa temperatura virtual em alguns índices, outra parcela de referência,
  outro método de integração);
* diferenças grandes indicam que algo deve ser investigado — **nunca** se copia o número do Wyoming para a apresentação:
  todo número usado vem das contas deste notebook (`metricas.json`), e a tabela abaixo registra a conferência.

In [ ]:
# Correspondência por palavras-chave (os nomes exatos variam entre as versões do site do Wyoming).
# A ordem importa: a 1ª regra que casar define o índice deste notebook (None = sem correspondente).
CORRESP = [
    (('showalter',), 'Showalter_K'),
    (('lifted index',), 'LI_K'), (('lift ',), 'LI_K'),
    (('sweat',), 'SWEAT'),
    (('k index',), 'K_index_C'),
    (('cross totals',), None), (('vertical totals',), None), (('totals',), 'TotalTotals_C'),
    (('cape',), 'SBCAPE_Jkg'), (('convective available',), 'SBCAPE_Jkg'),
    (('cin',), 'SBCIN_Jkg'), (('convective inhibition',), 'SBCIN_Jkg'),
    (('lcl', 'pres'), 'LCL_hPa'), (('lifted condensation', 'pres'), 'LCL_hPa'),
    (('lfc',), 'LFC_hPa'), (('free convection',), 'LFC_hPa'),
    (('equilib',), 'EL_hPa'),
    (('precipitable',), 'PW_mm'),
]

def chave_notebook(nome_w):
    n = nome_w.lower() + ' '
    for palavras, chave in CORRESP:
        if all(p in n for p in palavras):
            return chave
    return None

def conferencia(c):
    W = c['indices_wyoming']; I = c['indices']
    linhas = []
    for nome_w, val_w in W.items():
        chave = chave_notebook(nome_w)
        calc = I.get(chave, np.nan) if chave else np.nan
        linhas.append(dict(indice_wyoming=nome_w, valor_wyoming=val_w, indice_notebook=chave or '—',
                           valor_notebook=calc, diferenca=calc - val_w if chave else np.nan))
    return pd.DataFrame(linhas)

for c in CASOS:
    print(f"\n=== {c['rotulo']} ===")
    if not c['indices_wyoming']:
        print('índices do Wyoming indisponíveis — conferência não feita')
        c['conferencia_wyoming'] = None
        continue
    tab = conferencia(c)
    c['conferencia_wyoming'] = tab.to_dict(orient='records')
    print('fonte:', c['fonte_indices_wyoming'])
    display(tab.round(2))

## 6. Skew-T e hodógrafo

Níveis apontados pelo QC aparecem marcados com **×** no Skew-T. A tabela abaixo de cada figura é montada **a partir do dicionário de índices** (não há valores digitados).

In [ ]:
def fmt(v, casas=0, unid=''):
    return 'n/d' if v is None or (isinstance(v, float) and np.isnan(v)) else f'{v:.{casas}f}{unid}'

def figura_skewt(c):
    P, I = c['perfis'], c['indices']
    p, T, Td, h, u, v = P['p'], P['T'], P['Td'], P['h'], P['u'], P['v']
    fig = plt.figure(figsize=(16, 10))
    skew = SkewT(fig, rotation=45, rect=[0.05, 0.30, 0.45, 0.62])
    skew.plot(p, T, 'r', lw=2, label='T')
    skew.plot(p, Td, 'g', lw=2, label='Td')
    skew.plot(p, P['perfil_sb'], 'k--', lw=1.5, label='parcela de superfície')
    if 'perfil_mu' in P and abs(I['MU_origem_hPa'] - p[0].m) > 1:
        skew.plot(P['p_mu'], P['perfil_mu'], color='purple', ls=':', lw=1.8,
                  label=f"parcela MU ({I['MU_origem_hPa']:.0f} hPa)")
    skew.shade_cape(p, T, P['perfil_sb'], alpha=0.3)
    skew.shade_cin(p, T, P['perfil_sb'], alpha=0.3)
    for k, cor in (('LCL', 'tab:blue'), ('LFC', 'tab:orange'), ('EL', 'tab:red')):
        if not np.isnan(I[f'{k}_hPa']):
            skew.ax.axhline(I[f'{k}_hPa'], color=cor, ls='--', lw=1, label=f"{k} {I[f'{k}_hPa']:.0f} hPa")
    if not c['qc'].empty:
        sus = c['df_original'][c['df_original']['p'].isin(c['qc']['p_hPa'])]
        skew.plot(sus['p'].values*units.hPa, sus['T'].values*units.degC, 'kx', ms=10, mew=2, label='nível apontado pelo QC')
    vis = p.m >= P_TOPO
    passo = max(1, int(vis.sum()) // 30)
    skew.plot_barbs(p[vis][::passo], u[vis][::passo], v[vis][::passo])
    skew.plot_dry_adiabats(lw=0.5); skew.plot_moist_adiabats(lw=0.5); skew.plot_mixing_lines(lw=0.5)
    skew.ax.set_ylim(1050, P_TOPO); skew.ax.set_xlim(-30, 45)
    skew.ax.legend(loc='upper left', fontsize=8)
    skew.ax.set_xlabel('temperatura (°C)'); skew.ax.set_ylabel('pressão (hPa)')
    skew.ax.set_title('Skew-T log-p', fontsize=13)

    ax = fig.add_axes([0.56, 0.36, 0.40, 0.52])
    um, vm, hk = u.to('m/s').m, v.to('m/s').m, h.to('km').m
    sel = p.m >= P_TOPO
    hodo = Hodograph(ax, component_range=max(20, np.nanmax(np.hypot(um[sel], vm[sel])) * 1.15))
    hodo.add_grid(increment=10)
    l = hodo.plot_colormapped(um[sel], vm[sel], hk[sel], cmap='viridis')
    plt.colorbar(l, ax=ax, label='altura AGL (km)', shrink=0.8)
    for alvo in (0.5, 1, 3, 6, 9):
        if alvo > hk[sel].max(): continue
        k = int(np.argmin(abs(hk - alvo)))
        if abs(hk[k] - alvo) < 0.6:
            ax.plot(um[k], vm[k], 'ko', ms=4); ax.annotate(f'{hk[k]:.1f} km', (um[k], vm[k]), fontsize=8,
                                                             xytext=(4, 4), textcoords='offset points')
    for nome, mk, cor in (('LM', 's', 'tab:blue'), ('RM', 'D', 'tab:red')):
        if f'Bunkers_{nome}_u_ms' in I:
            ax.plot(I[f'Bunkers_{nome}_u_ms'], I[f'Bunkers_{nome}_v_ms'], mk, color=cor, ms=8,
                    label=f'Bunkers {nome} ({I[f"Bunkers_{nome}_u_ms"]:.1f}, {I[f"Bunkers_{nome}_v_ms"]:.1f}) m/s')
    ax.legend(fontsize=8, loc='lower left'); ax.set_xlabel('u (m/s)'); ax.set_ylabel('v (m/s)')
    ax.set_title(f'Hodógrafo superfície–{P_TOPO} hPa (m/s)', fontsize=13)

    linhas = [
        ['SBCAPE ' + fmt(I['SBCAPE_Jkg'], 0, ' J/kg'), 'SBCIN ' + fmt(I['SBCIN_Jkg'], 0, ' J/kg'),
         'MLCAPE ' + fmt(I['MLCAPE_Jkg'], 0, ' J/kg'), 'MUCAPE ' + fmt(I['MUCAPE_Jkg'], 0, ' J/kg'),
         'MU origem ' + fmt(I['MU_origem_hPa'], 0, ' hPa'), 'DCAPE ' + fmt(I['DCAPE_Jkg'], 0, ' J/kg')],
        ['LCL ' + fmt(I['LCL_hPa'], 0, ' hPa'), 'LFC ' + fmt(I['LFC_hPa'], 0, ' hPa'), 'EL ' + fmt(I['EL_hPa'], 0, ' hPa'),
         'LI ' + fmt(I['LI_K'], 1, ' K'), 'Showalter ' + fmt(I['Showalter_K'], 1, ' K'), 'K ' + fmt(I['K_index_C'], 1, ' °C')],
        ['TT ' + fmt(I['TotalTotals_C'], 1, ' °C'), 'PW ' + fmt(I['PW_mm'], 1, ' mm'),
         'BS 0–1 km ' + fmt(I['BS0_1km_ms'], 1, ' m/s'), 'BS 0–3 km ' + fmt(I['BS0_3km_ms'], 1, ' m/s'),
         'BS 0–6 km ' + fmt(I['BS0_6km_ms'], 1, ' m/s'), 'STP fixo (HS) ' + fmt(I['STP_fixo_HS'], 2)],
        ['SRH 0–1 LM ' + fmt(I.get('SRH0_1km_LM_m2s2'), 0, ' m²/s²'), 'SRH 0–3 LM ' + fmt(I.get('SRH0_3km_LM_m2s2'), 0, ' m²/s²'),
         'SRH 0–1 RM ' + fmt(I.get('SRH0_1km_RM_m2s2'), 0, ' m²/s²'), 'SRH 0–3 RM ' + fmt(I.get('SRH0_3km_RM_m2s2'), 0, ' m²/s²'),
         'EHI 0–3 (HS) ' + fmt(I['EHI0_3_HS'], 2),
         f"vento máx <700 hPa {fmt(I['vento_max_abaixo700_kt'], 0, ' kt')} ({fmt(I['vento_max_abaixo700_dir'], 0, '°')}, {fmt(I['vento_max_abaixo700_p_hPa'], 0, ' hPa')})"],
    ]
    axt = fig.add_axes([0.03, 0.02, 0.94, 0.20]); axt.axis('off')
    tb = axt.table(cellText=linhas, loc='center', cellLoc='center'); tb.auto_set_font_size(False)
    tb.set_fontsize(9); tb.scale(1, 1.6)
    excl = f" | níveis excluídos: {c['niveis_excluidos']}" if c['niveis_excluidos'] else ''
    fig.suptitle(f"{c['rotulo']}{excl}\nFonte: {c['fonte']}", fontsize=12, y=1.0)
    arq = os.path.join(DIR_SAIDA, f"skewt_hodografo_{c['chave']}.png")
    fig.savefig(arq, dpi=150, bbox_inches='tight'); plt.show()
    return arq

for c in CASOS:
    c.setdefault('figuras', {})['skewt'] = figura_skewt(c)

## 7. Perfis de θ, θe, θes, N² e estabilidade estática

* θes = temperatura potencial equivalente **de saturação** (`saturation_equivalent_potential_temperature`).
* **N² = (g/θ) ∂θ/∂z** com a altura **observada**; N² < 0 indica camada estaticamente instável (para ar seco).
* Estabilidade estática **S = −(T/θ) ∂θ/∂p**, em K/hPa.

In [ ]:
def perfis_termo(df):
    p = df['p'].values*units.hPa; T = df['T'].values*units.degC; Td = df['Td'].values*units.degC
    z = df['z'].values*units.m
    th = mpcalc.potential_temperature(p, T)
    the = mpcalc.equivalent_potential_temperature(p, T, Td)
    thes = mpcalc.saturation_equivalent_potential_temperature(p, T)
    N2 = mpcalc.brunt_vaisala_frequency_squared(z, th)
    # S = -(T/θ) ∂θ/∂p (o static_stability do MetPy é outra grandeza: σ, em J kg⁻¹ hPa⁻²)
    S = -(T.to('K').m / th.m) * np.gradient(th.m, p.m)
    return dict(p=p.m, th=th.m, the=the.m, thes=thes.m, N2=N2.to('1/s^2').m, S=S, z=z.m)

def figura_perfis(c):
    Pf = perfis_termo(c['df']); c['perfis_termo'] = Pf
    fig, ax = plt.subplots(1, 3, figsize=(16, 7), sharey=True)
    ax[0].plot(Pf['th'], Pf['p'], label='θ'); ax[0].plot(Pf['the'], Pf['p'], label='θe')
    ax[0].plot(Pf['thes'], Pf['p'], label='θes'); ax[0].set_xlabel('K'); ax[0].legend()
    ax[0].set_xlim(280, 380)
    ax[1].plot(Pf['N2']*1e4, Pf['p'], 'k'); ax[1].axvline(0, color='r', lw=0.8)
    ax[1].set_xlabel('N² (10⁻⁴ s⁻²)'); ax[1].set_xlim(-5, 10)
    ax[2].plot(Pf['S'], Pf['p'], 'b'); ax[2].axvline(0, color='r', lw=0.8)
    ax[2].set_xlabel('S = −(T/θ)∂θ/∂p (K/hPa)'); ax[2].set_xlim(-0.2, 0.6)
    for a in ax:
        a.set_yscale('log'); a.set_ylim(1050, P_TOPO); a.grid(alpha=0.3)
        tk = [t for t in (1000, 925, 850, 700, 600, 500, 400, 300, 250, 200) if t >= P_TOPO]
        a.set_yticks(tk); a.set_yticklabels(map(str, tk)); a.minorticks_off()
        for ps in c['qc']['p_hPa']: a.axhline(ps, color='gray', ls=':', lw=1)
    ax[0].set_ylabel('pressão (hPa)')
    fig.suptitle(f"Perfis — {c['rotulo']} (linhas pontilhadas: níveis apontados pelo QC)")
    arq = os.path.join(DIR_SAIDA, f"perfis_{c['chave']}.png")
    fig.savefig(arq, dpi=150, bbox_inches='tight'); plt.show()
    return arq

for c in CASOS:
    c['figuras']['perfis'] = figura_perfis(c)

## 8. Comparação entre as sondagens

Figuras: θe e depressão do ponto de orvalho das sondagens sobrepostas, e **θ, θe e θes de cada sondagem lado a lado**
(mesmos eixos). Camadas com θe diminuindo com a altura indicam instabilidade convectiva (potencial); a distância entre
θe e θes indica o quanto o ar está longe da saturação.

A classificação abaixo usa um **critério explícito** (editável). Ela serve para conferir se as datas escolhidas
representam de fato regimes diferentes — se não representarem, **troque a data, não force a interpretação**.

In [ ]:
def classifica(I):
    # Critério simples e explícito — ajuste se quiser outro
    cape = np.nanmax([I['SBCAPE_Jkg'], I['MLCAPE_Jkg']])
    if cape < 100 and I['LI_K'] > 0:
        return 'estável (CAPE < 100 J/kg e LI > 0)'
    if cape < 1000:
        return 'marginal (100 ≤ CAPE < 1000 J/kg)'
    return 'instável (CAPE ≥ 1000 J/kg)'

for c in CASOS:
    c['classificacao'] = classifica(c['indices'])
    print(f"{c['rotulo']}: {c['classificacao']}")

fig, ax = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
for c in CASOS:
    Pf = c['perfis_termo']
    ax[0].plot(Pf['the'], Pf['p'], label=c['rotulo'])
    ax[1].plot(c['df']['T'] - c['df']['Td'], c['df']['p'], label=c['rotulo'])
ax[0].set_xlabel('θe (K)'); ax[1].set_xlabel('depressão do ponto de orvalho T − Td (°C)')
for a in ax:
    a.set_yscale('log'); a.set_ylim(1050, P_TOPO); a.grid(alpha=0.3); a.legend(fontsize=8)
    tk = [t for t in (1000, 925, 850, 700, 600, 500, 400, 300, 250, 200) if t >= P_TOPO]
    a.set_yticks(tk); a.set_yticklabels(map(str, tk)); a.minorticks_off()
ax[0].set_ylabel('pressão (hPa)')
arq_comp = os.path.join(DIR_SAIDA, 'comparacao_sondagens.png')
fig.savefig(arq_comp, dpi=150, bbox_inches='tight'); plt.show()

# θ, θe e θes das sondagens lado a lado (mesmos eixos para comparar)
vals = np.concatenate([np.concatenate([c['perfis_termo'][k][c['perfis_termo']['p'] >= P_TOPO]
                                       for k in ('th', 'the', 'thes')]) for c in CASOS])
x0, x1 = np.floor(np.nanmin(vals) / 5) * 5 - 2, np.ceil(np.nanmax(vals) / 5) * 5 + 2
fig, axs = plt.subplots(1, len(CASOS), figsize=(5.5 * len(CASOS), 7.5), sharey=True, squeeze=False)
for a, c in zip(axs[0], CASOS):
    Pf = c['perfis_termo']
    a.plot(Pf['th'], Pf['p'], color='tab:blue', lw=2, label='θ')
    a.plot(Pf['the'], Pf['p'], color='tab:red', lw=2, label='θe')
    a.plot(Pf['thes'], Pf['p'], color='tab:green', lw=2, ls='--', label='θes')
    for ps in c['qc']['p_hPa']:
        if ps >= P_TOPO: a.axhline(ps, color='gray', ls=':', lw=1)
    a.set_yscale('log'); a.set_ylim(1050, P_TOPO); a.set_xlim(x0, x1); a.grid(alpha=0.3)
    tk = [t for t in (1000, 925, 850, 700, 600, 500, 400, 300, 250, 200) if t >= P_TOPO]
    a.set_yticks(tk); a.set_yticklabels(map(str, tk)); a.minorticks_off()
    a.set_xlabel('temperatura potencial (K)'); a.set_title(c['rotulo'], fontsize=11)
    a.legend(loc='upper left', fontsize=9)
axs[0][0].set_ylabel('pressão (hPa)')
fig.suptitle(f'θ, θe e θes — superfície até {P_TOPO} hPa (pontilhado: níveis apontados pelo QC)', fontsize=13)
arq_theta = os.path.join(DIR_SAIDA, 'perfis_theta_sondagens.png')
fig.savefig(arq_theta, dpi=150, bbox_inches='tight'); plt.show()

comp = tabela.loc[['SBCAPE_Jkg', 'MLCAPE_Jkg', 'MUCAPE_Jkg', 'SBCIN_Jkg', 'LI_K', 'K_index_C', 'PW_mm',
                   'BS0_6km_ms', 'SRH0_3km_LM_m2s2', 'vento_max_abaixo700_kt']]
comp

## 9. Matrizes de diferença de temperatura de densidade — Kerry Emanuel (`wyoming.f`)

O programa Fortran de K. Emanuel (MIT, curso 12.811) levanta parcelas de **cada nível de origem** (eixo x) até cada nível (eixo y)
e calcula a diferença de **temperatura de densidade** parcela − ambiente para ascensão **reversível** (condensado fica na parcela)
e **pseudoadiabática** (condensado precipita). Referência: Emanuel (1994), *Atmospheric Convection*.

Aqui o `sounding.txt` lido pelo Fortran é **escrito pelo próprio notebook a partir da mesma sondagem** analisada
(incluindo as exclusões do QC), no formato de colunas fixas que o `wyoming.f` espera.

**Corte em −4 K.** O `wyoming.f` original grava as matrizes com `MAX(valor, -4.0)`: tudo abaixo de −4 K vira −4 K
(recurso para os gráficos do curso do MIT). Com `CORTE_EMANUEL_K = None` o notebook compila uma cópia do programa
**só sem essa linha** — as demais contas são idênticas — e as matrizes mostram a flutuabilidade negativa completa.
Nas regiões com valor > −4 K os dois resultados são iguais. A tabela de CAPE (`cape.out`) não é afetada.

In [ ]:
def prepara_wyoming_exe():
    pasta = os.path.join(DIR_SAIDA, 'emanuel'); os.makedirs(pasta, exist_ok=True)
    original = os.path.join(DIR_DADOS, 'wyoming.f')
    if not os.path.exists(original):
        import urllib.request
        urllib.request.urlretrieve('https://texmex.mit.edu/pub/emanuel/soundings/wyoming.f', original)
    codigo = open(original).read()
    linhas_corte = ['TRDBAR(I,J)=MAX(TRDBAR(I,J),-4.0)', 'TPDBAR(I,J)=MAX(TPDBAR(I,J),-4.0)']
    for l in linhas_corte:
        if l not in codigo:
            raise RuntimeError(f'Linha esperada não encontrada no wyoming.f: {l}')
    if CORTE_EMANUEL_K is None:
        for l in linhas_corte:
            codigo = codigo.replace(l, 'CONTINUE')          # remove só o corte; o resto é o original
    else:
        for l in linhas_corte:
            codigo = codigo.replace(l, l.replace('-4.0', f'{float(CORTE_EMANUEL_K):.1f}'))
    fonte = os.path.join(pasta, 'wyoming_usado.f')
    open(fonte, 'w').write(codigo)
    print('corte das matrizes:', 'nenhum' if CORTE_EMANUEL_K is None else f'{CORTE_EMANUEL_K} K', '| fonte compilada:', fonte)
    exe = os.path.abspath(os.path.join(pasta, 'wyoming.exe'))
    r = subprocess.run(['gfortran', '-O2', '-o', exe, fonte], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('Falha ao compilar wyoming.f:\n' + r.stderr)
    return exe

def escreve_sounding_txt(df, caso, arquivo):
    # Formato lido pelo wyoming.f: 10 linhas de cabeçalho; dados FORMAT(1X,F6.1,9X,F5.1,2X,F5.1) = p, T, Td
    with open(arquivo, 'w') as f:
        f.write('\n' * 10)
        for _, r in df.iterrows():
            f.write(f"{r['p']:7.1f}{r['z']:7.0f}{r['T']:7.1f}{r['Td']:7.1f}\n")
        f.write('</PRE>\n')
        f.write(f"Station identifier: {caso['estacao']}\n")
        f.write(f"Station number: {caso['estacao']}\n")
        f.write(f"Observation time: {caso['dt']:%y%m%d}/{caso['dt']:%H%M}\n")

def roda_emanuel(c, exe):
    pasta = os.path.join(DIR_SAIDA, 'emanuel', c['chave']); os.makedirs(pasta, exist_ok=True)
    escreve_sounding_txt(c['df'], c, os.path.join(pasta, 'sounding.txt'))
    r = subprocess.run([exe], cwd=pasta, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"wyoming.exe falhou ({c['rotulo']}):\n{r.stdout}\n{r.stderr}")
    E = dict(p=np.loadtxt(os.path.join(pasta, 'p.out')),
             porig=np.loadtxt(os.path.join(pasta, 'porig.out')),
             rev=np.loadtxt(os.path.join(pasta, 'tdifrev.out')),
             pse=np.loadtxt(os.path.join(pasta, 'tdifpseudo.out')))
    # tabela cape.out (J/kg): origem, PA rev, PA pse, NA rev, NA pse, CAPE rev, CAPE pse, DCAPE, ...
    linhas = [l.split() for l in open(os.path.join(pasta, 'cape.out')).read().splitlines()
              if re.match(r'^\s*-?\d+\.\d', l)]
    cape = pd.DataFrame([[float(x) for x in l[:8]] for l in linhas],
                        columns=['p_origem', 'PA_rev', 'PA_pse', 'NA_rev', 'NA_pse', 'CAPE_rev', 'CAPE_pse', 'DCAPE'])
    E['cape'] = cape
    return E

def figura_emanuel(c, E):
    X, Y = np.meshgrid(E['porig'], E['p'])
    fis = Y < X                                     # nível final acima da origem (o resto não tem sentido físico)
    vis = fis & (Y >= P_TOPO_EMANUEL)
    # Escala simétrica (branco = 0 K) ajustada à flutuabilidade POSITIVA, mínimo ±6 K;
    # valores mais negativos (p.ex. estratosfera) saturam em azul escuro.
    amp = 2 * float(np.ceil(max(np.nanmax(E['rev'][vis]), np.nanmax(E['pse'][vis]), 6) / 2))   # par → isolinhas em 0, ±2, ±4…
    niveis = np.arange(-amp, amp + 0.25, 0.5)
    fig, axs = plt.subplots(1, 2, figsize=(16, 7.5), sharey=True)
    for ax, campo, nome in ((axs[0], E['rev'], 'reversível'), (axs[1], E['pse'], 'pseudoadiabática')):
        Z = np.where(fis, campo, np.nan)
        cf = ax.contourf(X, Y, Z, levels=niveis, cmap='RdBu_r', vmin=-amp, vmax=amp, extend='both')
        cs = ax.contour(X, Y, Z, levels=[v for v in np.arange(-amp, amp + 0.1, 2) if v != 0],
                        colors='k', linewidths=0.7)              # negativos tracejados
        ax.clabel(cs, fmt='%d', fontsize=8)
        cz = ax.contour(X, Y, Z, levels=[0], colors='k', linewidths=2.0)
        ax.clabel(cz, fmt='0', fontsize=9)
        ax.set_xlim(E['porig'].max(), E['porig'].min()); ax.set_ylim(E['p'].max(), P_TOPO_EMANUEL)
        ax.xaxis.set_major_formatter(plt.FormatStrFormatter('%.0f'))
        ax.set_xlabel('pressão de origem da parcela (hPa)'); ax.set_title(f'ascensão {nome}')
        ax.grid(alpha=0.3, ls=':')
    axs[0].set_ylabel('pressão até onde a parcela é levantada (hPa)')
    corte = 'sem corte' if CORTE_EMANUEL_K is None else f'valores < {CORTE_EMANUEL_K} K truncados (como no original)'
    fig.colorbar(cf, ax=axs, orientation='horizontal', fraction=0.05, pad=0.1, ticks=np.arange(-amp, amp + 1, 2),
                 label=f'diferença de temperatura de densidade parcela − ambiente (K) — {corte}')
    k = E['cape']['CAPE_pse'].idxmax()
    fig.suptitle(f"Emanuel (wyoming.f) — {c['rotulo']}\nCAPE pseudoadiabática máx. = {E['cape'].loc[k,'CAPE_pse']:.0f} J/kg "
                 f"(origem {E['cape'].loc[k,'p_origem']:.0f} hPa); da superfície = {E['cape'].iloc[0]['CAPE_pse']:.0f} J/kg")
    arq = os.path.join(DIR_SAIDA, f"emanuel_{c['chave']}.png")
    fig.savefig(arq, dpi=150, bbox_inches='tight'); plt.show()
    return arq

exe = prepara_wyoming_exe()
for c in CASOS:
    E = roda_emanuel(c, exe); c['emanuel'] = E
    c['figuras']['emanuel'] = figura_emanuel(c, E)
    cp = E['cape']; k = cp['CAPE_pse'].idxmax()
    c['indices_emanuel'] = dict(
        CAPE_rev_superficie_Jkg=float(cp.iloc[0]['CAPE_rev']), CAPE_pse_superficie_Jkg=float(cp.iloc[0]['CAPE_pse']),
        CAPE_rev_max_Jkg=float(cp['CAPE_rev'].max()), CAPE_pse_max_Jkg=float(cp.loc[k, 'CAPE_pse']),
        CAPE_pse_max_origem_hPa=float(cp.loc[k, 'p_origem']), DCAPE_max_Jkg=float(cp['DCAPE'].max()))
    print(c['rotulo'], c['indices_emanuel'])

## 10. Satélite (opcional)

**GridSat-B1** (NOAA/NCEI CDR, Knapp et al. 2011): **temperatura de brilho no IR janela (~11 µm)**, grade de **0,07° (~8 km)**, a cada 3 h.
O notebook baixa **só o recorte da região** por OPeNDAP (THREDDS da NCEI) e salva em `metpack/sat/` — nas próximas execuções usa o
arquivo salvo, sem internet. (Produtos em grade de 1°, como o ISCCP-H HGG, não são usados: resolução insuficiente para mesoescala.)

Contornos (costa, países, estados) vêm de `metpack/mapas/` (Natural Earth 10m recortado para a América do Sul), então os mapas
**não dependem de download**. O mínimo de temperatura citado em cada painel é **calculado** dentro da caixa tracejada.

In [ ]:
import xarray as xr
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader

DATA_SAT = [datetime(1995, 12, 24, 12), datetime(1995, 12, 24, 18)]
EXTENSAO = [-62, -42, -36, -21]                    # lon0, lon1, lat0, lat1 do mapa
CAIXA = dict(lon=(-58, -46), lat=(-33, -25))       # caixa para o mínimo (RS/SC e oceano adjacente)
DIR_MAPAS = os.path.join(DIR_DADOS, 'mapas')
PC = ccrs.PlateCarree()

def mapa_base(ax, cor='k'):
    # Contornos locais (Natural Earth 10m recortado). Se a pasta não existir, tenta o download do cartopy.
    camadas = [('costa', 0.9), ('paises', 0.9), ('estados', 0.5)]
    if all(os.path.exists(os.path.join(DIR_MAPAS, f'{n}.shp')) for n, _ in camadas):
        for n, lw in camadas:
            geoms = list(shpreader.Reader(os.path.join(DIR_MAPAS, f'{n}.shp')).geometries())
            ax.add_geometries(geoms, PC, facecolor='none', edgecolor=cor, linewidth=lw,
                              linestyle='-' if n != 'estados' else (0, (4, 2)))
    else:
        import cartopy.feature as cfeature
        print(f'⚠ {DIR_MAPAS} não encontrado — usando download do cartopy (precisa de internet)')
        ax.add_feature(cfeature.COASTLINE.with_scale('10m'), edgecolor=cor, lw=0.9)
        ax.add_feature(cfeature.BORDERS.with_scale('10m'), edgecolor=cor, lw=0.9)
        ax.add_feature(cfeature.STATES.with_scale('10m'), edgecolor=cor, lw=0.5)
    g = ax.gridlines(PC, draw_labels=True, lw=0.3, alpha=0.5, color='gray')
    g.top_labels = g.right_labels = False

def cmap_ir_realcado():
    # Realce IR usual: tons de cinza para topos quentes (> -30 °C) e cores para topos frios.
    # Escala de -90 a +40 °C (130 níveis de 1 K).
    cinza = plt.get_cmap('gray_r')(np.linspace(0.15, 0.9, 70))            # -30 → +40: claro → escuro
    frio = plt.get_cmap('jet')(np.linspace(0.05, 1.0, 60))[::-1]          # -30 → -90
    cores = np.vstack([frio, cinza])                                       # índice 0 = -90 °C
    cm = mcolors.ListedColormap(cores, name='ir_realcado'); cm.set_bad('#f2c4c4')   # sem dado = rosa
    return cm, mcolors.Normalize(vmin=-90, vmax=40)

def recorte(ds):
    lat = ds['lat']
    return ds.sel(lon=slice(EXTENSAO[0] - 1, EXTENSAO[1] + 1),
                  lat=slice(EXTENSAO[2] - 1, EXTENSAO[3] + 1) if lat[0] < lat[-1] else slice(EXTENSAO[3] + 1, EXTENSAO[2] - 1))

def gridsat(dt):
    nome = f'GRIDSAT-B1.{dt:%Y.%m.%d.%H}.v02r01.nc'
    local = os.path.join(DIR_DADOS, 'sat', nome.replace('.nc', '_recorte_sul.nc'))
    if not os.path.exists(local):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        erros = []
        fontes = [f'https://www.ncei.noaa.gov/thredds/dodsC/cdr/gridsat/{dt:%Y}/{nome}',     # OPeNDAP: só o recorte
                  f'https://www.ncei.noaa.gov/data/geostationary-ir-channel-brightness-temperature-gridsat-b1/access/{dt:%Y}/{nome}']
        for url in fontes:
            try:
                if 'dodsC' in url:
                    ds = xr.open_dataset(url)
                else:
                    import urllib.request, tempfile
                    tmp = os.path.join(tempfile.gettempdir(), nome)
                    urllib.request.urlretrieve(url, tmp)
                    ds = xr.open_dataset(tmp)
                recorte(ds[['irwin_cdr']]).load().to_netcdf(local)
                print('baixado:', url, '→', local)
                break
            except Exception as e:
                erros.append(f'{url}: {type(e).__name__}: {e}')
        else:
            raise RuntimeError('GridSat-B1 indisponível. Baixe manualmente o arquivo\n  ' + fontes[1] +
                               f'\ne salve em {os.path.join(DIR_DADOS, "sat", nome)} (o notebook recorta e usa).\n' + '\n'.join(erros))
    ds = xr.open_dataset(local)
    tb = ds['irwin_cdr'].squeeze()
    tb = tb.where((tb > 150) & (tb < 340)) - 273.15
    return tb.sortby('lat'), f'GridSat-B1 v02r01 (NOAA/NCEI), 0,07° — {nome}'

def gridsat_arquivo_global_manual(dt):
    # Se o usuário baixou o arquivo global à mão, gera o recorte a partir dele.
    nome = f'GRIDSAT-B1.{dt:%Y.%m.%d.%H}.v02r01.nc'
    glob_arq = os.path.join(DIR_DADOS, 'sat', nome)
    local = glob_arq.replace('.nc', '_recorte_sul.nc')
    if os.path.exists(glob_arq) and not os.path.exists(local):
        recorte(xr.open_dataset(glob_arq)[['irwin_cdr']]).load().to_netcdf(local)

def minimo_caixa(da):
    s = da.sel(lon=slice(*CAIXA['lon']), lat=slice(*CAIXA['lat']))
    if int(s.notnull().sum()) == 0: return np.nan, np.nan, np.nan
    m = s.where(s == s.min(), drop=True)
    return float(s.min()), float(m.lat[0]), float(m.lon[0])

def figura_satelite(produto, leitor):
    cm, norm = cmap_ir_realcado()
    fig, axs = plt.subplots(1, 2, figsize=(17, 6.8), subplot_kw={'projection': PC})
    resultados = {}
    for ax, dt in zip(axs, DATA_SAT):
        campo, fonte = leitor(dt)
        im = ax.pcolormesh(campo.lon, campo.lat, campo, cmap=cm, norm=norm, shading='nearest', transform=PC)
        ax.set_extent(EXTENSAO, crs=PC)
        mapa_base(ax, cor='white')
        ax.plot(-51.18, -30.0, '^', color='yellow', ms=9, mec='k', transform=PC)
        ax.text(-50.9, -29.8, 'SBPA', fontsize=9, color='yellow', fontweight='bold', transform=PC)
        x0, x1 = CAIXA['lon']; y0, y1 = CAIXA['lat']
        ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], '--', color='cyan', lw=1, transform=PC)
        tmin, la, lo = minimo_caixa(campo)
        resultados[f'{dt:%Y%m%d_%H}'] = dict(Tmin_C=tmin, lat=la, lon=lo, fonte=fonte)
        if not np.isnan(tmin):
            ax.plot(lo, la, 'x', color='magenta', ms=10, mew=2, transform=PC)
        ax.set_title(f'{dt:%d/%m/%Y %H} UTC — mínimo na caixa: {tmin:.1f} °C em ({la:.2f}, {lo:.2f})', fontsize=10)
        print(fonte)
    fig.colorbar(im, ax=axs, orientation='horizontal', fraction=0.045, pad=0.07, extend='both',
                 ticks=np.arange(-90, 41, 10),
                 label='°C — temperatura de brilho IR (~11 µm)')
    fig.suptitle('GridSat-B1 (NOAA/NCEI) — temperatura de brilho no IR janela, 0,07°', fontsize=13, y=0.97)
    arq = os.path.join(DIR_SAIDA, f'satelite_{produto}.png')
    fig.savefig(arq, dpi=200, bbox_inches='tight'); plt.show()
    return arq, resultados

SATELITE = {}
if FAZER_SATELITE:
    for dt in DATA_SAT:
        gridsat_arquivo_global_manual(dt)
    for produto, leitor in (('gridsat', gridsat),):
        try:
            arq, res = figura_satelite(produto, leitor)
            SATELITE[produto] = dict(figura=arq, resultados=res)
        except Exception as e:
            print(f'⚠ {produto} não gerado: {type(e).__name__}: {e}')
SATELITE

## 11. Carta sinótica com reanálise (opcional)

**NCEP/NCAR Reanálise 1** (Kalnay et al. 1996), grade 2,5°, lida por OPeNDAP no PSL/NOAA:
geopotencial em 500 hPa, vento e θe em 850 hPa no horário da sondagem. **Nada é desenhado à mão.**
Para mais resolução, troque por ERA5 (precisa de conta no Copernicus CDS).

In [ ]:
DATA_SINOTICA = datetime(1995, 12, 24, 12)

def carta_sinotica(dt):
    import xarray as xr, cartopy.crs as ccrs
    base = 'https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis/pressure'
    def campo(var, nivel):
        ds = xr.open_dataset(f'{base}/{var}.{dt:%Y}.nc')
        da = ds[var].sel(time=dt, level=nivel)
        da = da.assign_coords(lon=((da.lon + 180) % 360) - 180).sortby('lon').sortby('lat')
        return da.sel(lon=slice(-90, -20), lat=slice(-55, -5))
    z500 = campo('hgt', 500); u850 = campo('uwnd', 850); v850 = campo('vwnd', 850)
    t850 = campo('air', 850); q850 = campo('shum', 850)
    td850 = mpcalc.dewpoint_from_specific_humidity(850*units.hPa, t850.values*units.K, q850.values*units('kg/kg'))
    the850 = mpcalc.equivalent_potential_temperature(850*units.hPa, t850.values*units.K, td850).m
    fig, axs = plt.subplots(1, 2, figsize=(16, 7.5), subplot_kw={'projection': ccrs.PlateCarree()})
    cs = axs[0].contour(z500.lon, z500.lat, z500, levels=np.arange(5400, 6000, 30), colors='k', linewidths=1)
    axs[0].clabel(cs, fmt='%d', fontsize=8); axs[0].set_title(f'Z 500 hPa (gpm) — {dt:%d/%m/%Y %HZ}')
    cf = axs[1].contourf(t850.lon, t850.lat, the850, levels=np.arange(300, 366, 4), cmap='Spectral_r')
    axs[1].barbs(u850.lon[::2], u850.lat[::2], (u850.values*units('m/s')).to('kt').m[::2, ::2],
                 (v850.values*units('m/s')).to('kt').m[::2, ::2], length=5)
    fig.colorbar(cf, ax=axs[1], orientation='horizontal', fraction=0.05, pad=0.08, label='θe 850 hPa (K)')
    axs[1].set_title(f'θe e vento (kt) em 850 hPa — {dt:%d/%m/%Y %HZ}')
    for ax in axs:
        ax.set_extent([-80, -30, -50, -10]); mapa_base(ax)
        ax.plot(-51.18, -30.0, 'r*', ms=12, transform=ccrs.PlateCarree())
    fig.suptitle('Reanálise NCEP/NCAR R1 (PSL/NOAA, OPeNDAP)')
    arq = os.path.join(DIR_SAIDA, f'sinotica_ncep_{dt:%Y%m%d_%H}.png')
    fig.savefig(arq, dpi=150, bbox_inches='tight'); plt.show()
    return arq

SINOTICA = None
if FAZER_REANALISE:
    try:
        SINOTICA = carta_sinotica(DATA_SINOTICA)
    except Exception as e:
        print(f'⚠ carta sinótica não gerada (sem acesso ao OPeNDAP do PSL?): {type(e).__name__}: {e}')

## 12. Exportar `metricas.json`

Arquivo único com **todos os números** da análise. Apresentação, dashboard e roteiro devem **ler** daqui.

In [ ]:
def limpa_json(o):
    if isinstance(o, dict): return {k: limpa_json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [limpa_json(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else round(float(o), 3)
    if isinstance(o, np.integer): return int(o)
    return o

metricas = dict(
    gerado_em=datetime.now().isoformat(timespec='seconds'),
    metpy=metpy.__version__,
    convencoes=dict(alturas='AGL', movimento_tempestade='Bunkers et al. (2000); HS usa LM',
                    SRH='total (positiva+negativa); no HS SRH ciclônica é negativa',
                    camada_ML_hPa=CAMADA_ML_HPA, corte_matrizes_emanuel_K=CORTE_EMANUEL_K),
    casos={},
    figuras_comparacao=dict(theta_thetae_thetaes=arq_theta, thetae_e_depressao=arq_comp),
    satelite={k: v['resultados'] for k, v in SATELITE.items()},
    figura_sinotica=SINOTICA,
)
for c in CASOS:
    metricas['casos'][c['chave']] = dict(
        rotulo=c['rotulo'], estacao=c['estacao'], fonte=c['fonte'],
        n_niveis=len(c['df']), niveis_descartados_por_dado_faltante=c['n_descartados'],
        qc_niveis_suspeitos=c['qc'].to_dict(orient='records'), niveis_excluidos=c['niveis_excluidos'],
        classificacao=c['classificacao'], indices=c['indices'],
        indices_sem_exclusao=c.get('indices_sem_exclusao'), indices_emanuel=c.get('indices_emanuel'),
        indices_wyoming=c['indices_wyoming'], fonte_indices_wyoming=c['fonte_indices_wyoming'],
        conferencia_wyoming=c.get('conferencia_wyoming'),
        erros_de_calculo=c['erros'], figuras=c['figuras'])

with open(os.path.join(DIR_SAIDA, 'metricas.json'), 'w', encoding='utf-8') as f:
    json.dump(limpa_json(metricas), f, ensure_ascii=False, indent=2)
tabela.to_csv(os.path.join(DIR_SAIDA, 'metricas_indices.csv'))
print('salvo:', os.path.join(DIR_SAIDA, 'metricas.json'))
print(sorted(os.listdir(DIR_SAIDA)))

## 13. Para a interpretação (alunos)

* Toda afirmação na apresentação deve citar **um número de `metricas.json`** ou **uma figura deste notebook**.
* Afirmações sobre o **evento** (chuva observada, enchente, granizo, rajadas, danos) precisam de **fonte externa citada**
  (boletim do INMET, artigo, notícia de época). A sondagem sozinha **não** mostra que houve supercélula.
* Os valores oficiais vêm da sondagem completa. Se o QC apontar níveis, você pode mostrar um teste de sensibilidade (com e sem esses níveis) e discutir a diferença — sem afirmar erro de medição sem evidência.
* No Hemisfério Sul: rotação ciclônica = **horária**, SRH ciclônica **negativa**, supercélula favorecida = **left-mover**.

**Perguntas-guia**

1. Qual parcela (superfície, camada de mistura, mais instável) é mais representativa em cada caso? Por quê?
2. As matrizes de Emanuel mostram flutuabilidade positiva para parcelas de quais níveis de origem? Isso concorda com a MUCAPE?
3. A diferença entre ascensão reversível e pseudoadiabática é grande? Em que níveis?
4. O cisalhamento 0–6 km e a SRH 0–3 km (LM) são compatíveis com convecção organizada? Compare com limiares da literatura **citando a fonte**.